# Data generation for the Gray-Scott system

This notebook is based on https://github.com/wigging/gray-scott/blob/main/python-examples/grayscott_slices.py.

Boundary conditions have been modified relative to the original code

## Function definitions

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
phy_params = dict(
    Du = 0.2,    # diffusion coefficient for U
    Dv = 0.1,    # diffusion coefficient for V
    F = 0.035,   # feed rate
    k = 0.060,   # rate constant

    h = 2,       # grid spacing

    nt = 5003,  # number of time steps
    dt = 1,      # time step
)

def lap5(f, h2):
    """
    Use a five-point stencil with neumann boundary conditions to approximate
    the Laplacian. The corresponding array slices for each component of the
    stencil are noted.
    """

    left = f[1:-1, :-2]     # shift left for f(x - h, y)
    right = f[1:-1, 2:]     # shift right for f(x + h, y)
    down = f[2:, 1:-1]      # shift down for f(x, y - h)
    up = f[:-2, 1:-1]       # shift up for f(x, y + h)
    center = f[1:-1, 1:-1]  # center for f(x, y)

    fxy = (left + right + down + up - 4 * center) / h2

    fxy = np.pad(fxy, 1, mode='constant', constant_values=0) #Overriden after

    return fxy

def solve_grey_scott(
    n,                       # grid resolution
    sources_locations,       # list of (x,y) sources locations
    sources_width,           # width of the sources (int)
    Du, Dv, F, k, h, nt, dt,  # all phy params
    save_freqs = 1
    ):

    h2 = h * h

    # initialize arrays
    U = np.ones((n, n))
    V = np.zeros((n, n))

    for x,y in sources_locations:
        w2 = sources_width // 2
        U[x-w2:x+w2, y-w2:y+w2] = 0.5 + np.random.uniform(0, 0.1, (2*w2,2*w2))
        V[x-w2:x+w2, y-w2:y+w2] = 0.25 + np.random.uniform(0, 0.1, (2*w2,2*w2))

    history = {
        'U':[],
        'V':[]
    }

    # solve at each time step
    for it in range(nt):

        if it % save_freqs == 0:
            history['U'].append(U)
            history['V'].append(V)

        UVV = U * V * V
        U = U + (Du * lap5(U, h2) - UVV + F * (1 - U)) * dt
        V = V + (Dv * lap5(V, h2) + UVV - V * (F + k)) * dt

        U[0,:] = U[1,:]
        U[-1,:] = U[-2,:]
        U[:,0] = U[:,1]
        U[:,-1] = U[:,-2]

        V[0,:] = V[1,:]
        V[-1,:] = V[-2,:]
        V[:,0] = V[:,1]
        V[:,-1] = V[:,-2]

    history['U'] = np.stack(history['U'], axis = 0)
    history['V'] = np.stack(history['V'], axis = 0)

    return history


def generate_dataset(
        scale : int,
        n_simus : int,
        phy_params : dict,
        save_path: str,
        save_name: str,
        scale_1_n = 64,
        scale_1_n_sources = 3,
        sources_width = 10,
        save_freqs = 500,
        seed = None
    ):

    n = scale_1_n * scale
    n_sources = scale_1_n_sources * scale**2

    #sample sources positions
    gen = np.random.default_rng(seed)
    sources = gen.random((n_simus, n_sources, 2)) * (n - sources_width) + sources_width/2
    sources = np.array(sources, dtype = int)

    all_U = []
    all_V = []

    #run simulations
    for idx in tqdm(range(n_simus)):
        hist = solve_grey_scott(
            n = n,
            sources_locations = sources[idx],
            sources_width=sources_width,
            **phy_params,
            save_freqs = save_freqs
        )
        all_U.append(hist['U'])
        all_V.append(hist['V'])

    all_U = np.stack(all_U, axis = 0)
    all_V = np.stack(all_V, axis = 0)

    #position: regular grid of step h
    x = y = np.arange(n) * phy_params['h']
    x,y = np.meshgrid(x,y)
    pos = np.stack([x,y], axis = -1)
    
    #save
    os.makedirs(save_path, exist_ok=True)
    np.save(save_path + os.sep + f'{save_name}_pos.npy', pos)
    np.save(save_path + os.sep + f'{save_name}_U.npy', all_U)
    np.save(save_path + os.sep + f'{save_name}_V.npy', all_V)

# Generate

In [ ]:
scales = [1,2,3,7,10]
n_simus_per_scale = [1000, 50, 50, 25, 10]

for scale,n_simus in zip(scales, n_simus_per_scale):
    print(f'Generating scale {scale}')
    generate_dataset(
        scale, n_simus,
        phy_params,
        save_path='../data/academic_datasets/gray_scott',
        save_name=f's{scale}',
        seed = scale
    )

Generating scale 1


100%|██████████| 1000/1000 [07:48<00:00,  2.13it/s]


Generating scale 2


100%|██████████| 50/50 [00:54<00:00,  1.09s/it]


Generating scale 3


100%|██████████| 50/50 [01:40<00:00,  2.00s/it]


Generating scale 7


100%|██████████| 25/25 [05:01<00:00, 12.07s/it]


Generating scale 10


100%|██████████| 10/10 [04:37<00:00, 27.77s/it]
